In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats
import warnings
warnings.filterwarnings("ignore")


df = pd.read_csv("FEATURES_PREPARED.csv")
df["Date"] = pd.to_datetime(df["Date"], utc=True)
df = df.sort_values("Date").reset_index(drop=True)

print(f"Dataset: {df.shape[0]} rows, "
      f"{df['Date'].iloc[0].date()} → {df['Date'].iloc[-1].date()}")
print(f"Columns available: {list(df.columns)}\n")



PM_FEATURES = [
    "FED_DELTA",
    "GDP_DELTA",
    "UNEMPLOYMENT_DELTA",
    "INF_MONTHLY_DELTA",
]
PM_MISSING           = [f"{f}_is_missing" for f in PM_FEATURES]
ANNOUNCEMENT_DUMMIES = ["Ann_CPI", "Ann_FOMC", "Ann_Employment", "Ann_GDP"]

BASE_CONTROLS = [
    "VIX_chg",
    "DXY_chg",
    "US2Y_chg",
    "US10Y_chg",
    "hour_utc",
    "dow_utc",
] + ANNOUNCEMENT_DUMMIES

ASSETS_CONFIG = {
    "BTC":       "BTC_chg",
    "Gold":      "Gold_chg",
    "Oil":       "Oil_chg",
    "SPY":       "SPY_chg",
    "QQQ":       "QQQ_chg",
    "SP500_fut": "SP500_fut_chg",
}


def run_ols_hac(data, y_col, x_cols, hac_lags=5):
    """OLS with Newey-West HAC standard errors (handles serial correlation)."""
    temp = data[[y_col] + x_cols].dropna().copy()
    X    = sm.add_constant(temp[x_cols])
    y    = temp[y_col]
    return sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": hac_lags})


def delta_r2_ftest(baseline_model, augmented_model, pm_feature_names):
    """ΔR² and joint F-test: H0 = all PM coefficients = 0."""
    delta_r2 = augmented_model.rsquared - baseline_model.rsquared
    present  = [f for f in pm_feature_names if f in augmented_model.params.index]
    if not present:
        return delta_r2, None
    param_names = list(augmented_model.params.index)
    R = np.zeros((len(present), len(param_names)))
    for i, feat in enumerate(present):
        R[i, param_names.index(feat)] = 1.0
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        f_test = augmented_model.f_test(R)
    return delta_r2, f_test


def print_comparison(asset, baseline, augmented):
    dr2, ftest = delta_r2_ftest(baseline, augmented, PM_FEATURES)
    print(f"\n{'='*60}")
    print(f"  {asset} — OLS Results")
    print(f"{'='*60}")
    print(f"  Baseline  R²  : {baseline.rsquared:.6f}  (N={int(baseline.nobs)})")
    print(f"  Augmented R²  : {augmented.rsquared:.6f}")
    print(f"  ΔR²           : {dr2:.6f}")
    if ftest is not None:
        fval  = float(np.atleast_1d(ftest.fvalue).flat[0])
        fp    = float(ftest.pvalue)
        stars = "**" if fp < 0.05 else "*" if fp < 0.10 else ""
        print(f"  Joint F-test on PM block: F={fval:.4f}, p={fp:.4f} {stars}")
    print("\n  PM signal coefficients (augmented model):")
    for feat in PM_FEATURES:
        if feat in augmented.params.index:
            b = augmented.params[feat]
            p = augmented.pvalues[feat]
            s = "***" if p<0.01 else "**" if p<0.05 else "*" if p<0.10 else ""
            print(f"    {feat:30s}  β={b:+.6f}  p={p:.4f} {s}")


# OLS for all assets
print("\n" + "="*60)
print("  RUNNING OLS FOR ALL ASSETS")
print("="*60)

ols_models   = {}
summary_rows = []

for asset_name, col in ASSETS_CONFIG.items():

    if col not in df.columns:
        print(f"\n  SKIPPING {asset_name}: column '{col}' not found in data")
        continue

    print(f"\n--- {asset_name} ---")
    d            = df.copy()
    d["y"]       = d[col].shift(-1)
    d["own_lag1"]= d[col].shift(1)

    controls = ["own_lag1"] + BASE_CONTROLS

    baseline  = run_ols_hac(d, "y", controls)
    augmented = run_ols_hac(d, "y", controls + PM_FEATURES + PM_MISSING)

    print_comparison(asset_name, baseline, augmented)
    ols_models[asset_name] = (baseline, augmented, d)

    dr2, ft = delta_r2_ftest(baseline, augmented, PM_FEATURES)
    fp = float(ft.pvalue) if ft is not None else float("nan")
    summary_rows.append({
        "asset": asset_name, "col": col,
        "r2_base": round(baseline.rsquared, 6),
        "r2_aug":  round(augmented.rsquared, 6),
        "delta_r2": round(dr2, 6),
        "f_pvalue": round(fp, 4),
        "nobs": int(augmented.nobs)
    })


# Distributed Lag Model

print("\n\n" + "="*60)
print("  DISTRIBUTED LAG MODEL (DLM) — K=6 lags per PM signal")
print("="*60)

K = 6

def build_dlm(df_in, target_col, own_col, k=6):
    """Build DLM: target r(t+1) ~ own_lag + base_controls + PM_lags_1..k"""
    d            = df_in.copy()
    d["y"]       = d[target_col].shift(-1)
    d["own_lag1"]= d[own_col].shift(1)
    lagged_cols  = []
    for feat in PM_FEATURES:
        for lag in range(1, k+1):
            c = f"{feat}_lag{lag}"
            d[c] = d[feat].shift(lag)
            lagged_cols.append(c)
    x_cols = ["own_lag1"] + BASE_CONTROLS + lagged_cols
    return d, x_cols


def print_impulse_response(model, asset, pm_signal, k=6):
    params = model.params
    pvals  = model.pvalues
    print(f"\n  Impulse response: {pm_signal} → {asset}")
    for lag in range(1, k+1):
        col   = f"{pm_signal}_lag{lag}"
        if col in params.index:
            s = "***" if pvals[col]<0.01 else "**" if pvals[col]<0.05 else "*" if pvals[col]<0.10 else ""
            print(f"    h={lag}: β={params[col]:+.7f}  p={pvals[col]:.4f} {s}")

DLM_PAIRS = {
    "BTC":       [("FED_DELTA", "BTC")],
    "Gold":      [("FED_DELTA", "Gold"), ("GDP_DELTA", "Gold")],
    "Oil":       [("INF_MONTHLY_DELTA", "Oil")],
    "SPY":       [("FED_DELTA", "SPY"), ("UNEMPLOYMENT_DELTA", "SPY")],
    "QQQ":       [("FED_DELTA", "QQQ"), ("UNEMPLOYMENT_DELTA", "QQQ")],
    "SP500_fut": [("FED_DELTA", "SP500_fut"), ("UNEMPLOYMENT_DELTA", "SP500_fut")],
}

dlm_models = {}

for asset_name, col in ASSETS_CONFIG.items():
    if col not in df.columns:
        continue
    d_dlm, x_cols = build_dlm(df, col, col, k=K)
    dlm            = run_ols_hac(d_dlm, "y", x_cols)
    dlm_models[asset_name] = dlm

    print(f"\n{asset_name} DLM — R²={dlm.rsquared:.6f}  N={int(dlm.nobs)}")
    pairs = DLM_PAIRS.get(asset_name, [("FED_DELTA", asset_name)])
    for pm_signal, _ in pairs:
        print_impulse_response(dlm, asset_name, pm_signal)


# Summary table

print("\n\n" + "="*60)
print("  SUMMARY TABLE — OLS Results (All Assets)")
print("="*60)
print(f"  {'Asset':10}  {'R²_base':>10}  {'R²_aug':>10}  {'ΔR²':>10}  {'F-p':>8}  {'N':>6}")
print(f"  {'-'*60}")
for row in summary_rows:
    print(f"  {row['asset']:10}  {row['r2_base']:10.6f}  {row['r2_aug']:10.6f}  "
          f"{row['delta_r2']:10.6f}  {row['f_pvalue']:8.4f}  {row['nobs']:>6}")

pd.DataFrame(summary_rows).to_csv("ols_summary_table.csv", index=False)
print("\nSummary saved to: ols_summary_table.csv")



Dataset: 9673 rows, 2025-03-05 → 2026-04-12
Columns available: ['Date', 'SPY_chg', 'QQQ_chg', 'Gold_chg', 'Oil_chg', 'DXY_chg', 'BTC_chg', 'VIX_chg', 'SP500_fut_chg', 'US2Y_chg', 'US10Y_chg', 'is_weekend', 'hour_utc', 'dow_utc', 'Ann_CPI', 'Ann_Employment', 'Ann_FOMC', 'Ann_GDP', 'FED_DELTA', 'GDP_DELTA', 'UNEMPLOYMENT_DELTA', 'INF_YEARLY_DELTA', 'INF_MONTHLY_DELTA', 'fed_market_start', 'fed_market_end', 'gdp_market_start', 'gdp_market_end', 'labor_market_start', 'labor_market_end', 'inf_yearly_market_start', 'inf_yearly_market_end', 'inf_monthly_market_start', 'inf_monthly_market_end', 'FED_DELTA_is_missing', 'GDP_DELTA_is_missing', 'UNEMPLOYMENT_DELTA_is_missing', 'INF_YEARLY_DELTA_is_missing', 'INF_MONTHLY_DELTA_is_missing']


  RUNNING OLS FOR ALL ASSETS

--- BTC ---

  BTC — OLS Results
  Baseline  R²  : 0.008400  (N=1902)
  Augmented R²  : 0.011241
  ΔR²           : 0.002841
  Joint F-test on PM block: F=0.3188, p=0.8655 

  PM signal coefficients (augmented model):
    FED_DELTA